## Concept 3 — Structured Output Agent

### What is it?
Force the agent to return a **typed Pydantic object** instead of free text.
Two-stage pattern: Stage 1 executes tools. Stage 2 forces the LLM to fill a schema
using the tool results. Output is directly usable in code — no string parsing needed.

### Pipeline
```
Query
  → Stage 1: Tool-calling LLM executes tools → collects results
  → Stage 2: Structured-output LLM fills Pydantic schema
  → Returns typed object: response.answer, response.tools_used, response.confidence
```

### Limitation (what Concept 4 fixes)
Still stateless — no memory of previous turns.
→ Concept 4 adds message history so the agent remembers across turns.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

llm = get_llm()
tool_map = {t.name: t for t in TOOLS}
print(f'LLM ready | Tools: {list(tool_map.keys())}')

### Step 1 — Define the Output Schema
The schema defines exactly what fields the agent must return.
Each field has a description the LLM reads to understand what to fill in.

In [ ]:
class AgentResponse(BaseModel):
    answer:     str        = Field(description='The complete answer to the user query')
    tools_used: List[str]  = Field(description='List of tool names that were called, e.g. ["calculate", "get_weather"]')
    confidence: str        = Field(description='Confidence level: high, medium, or low')
    reasoning:  str        = Field(description='One sentence explaining how you arrived at the answer')

print('Schema defined:', list(AgentResponse.model_fields.keys()))

### Step 2 — Stage 1: Execute Tools
The first LLM call uses `bind_tools` to collect tool results.

In [ ]:
llm_with_tools = llm.bind_tools(TOOLS)

def execute_tools(query: str):
    messages = [HumanMessage(content=query)]
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    used_tools = []
    tool_results = []

    for call in response.tool_calls:
        result = tool_map[call['name']].invoke(call['args'])
        used_tools.append(call['name'])
        tool_results.append(f'{call["name"]}({call["args"]}) = {result}')
        messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))
        print(f'  Tool: {call["name"]}({call["args"]}) → {result}')

    return messages, used_tools, tool_results

# Test
msgs, used, results = execute_tools('What is 144 divided by 12?')
print(f'Tools used: {used}')
print(f'Results: {results}')

### Step 3 — Stage 2: Structured Synthesis
The second LLM call uses `with_structured_output` to fill the Pydantic schema
using the tool results from Stage 1.

In [ ]:
structured_llm = llm.with_structured_output(AgentResponse)

synthesis_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are an assistant that synthesizes tool results into a structured response. '
     'Fill all fields accurately based on the tool results provided.'),
    ('human',
     'Original query: {query}\n\n'
     'Tool results:\n{tool_results}\n\n'
     'Tools used: {tools_used}\n\n'
     'Now fill the response schema.'),
])

synthesis_chain = synthesis_prompt | structured_llm

# Test synthesis on a simple query
query = 'What is 144 divided by 12?'
msgs, used, results = execute_tools(query)

structured_response = synthesis_chain.invoke({
    'query':        query,
    'tool_results': '\n'.join(results) if results else 'No tools used',
    'tools_used':   str(used),
})

print(f'\nType: {type(structured_response).__name__}')
print(f'answer:     {structured_response.answer}')
print(f'tools_used: {structured_response.tools_used}')
print(f'confidence: {structured_response.confidence}')
print(f'reasoning:  {structured_response.reasoning}')

### Step 4 — Compare: Unstructured vs Structured
Same query — see the difference between free text and typed object.

In [ ]:
q = 'What is the weather in Hyderabad?'

# Unstructured — from Concept 1
llm_with_tools2 = llm.bind_tools(TOOLS)
msgs2 = [HumanMessage(content=q)]
r2 = llm_with_tools2.invoke(msgs2)
msgs2.append(r2)
if r2.tool_calls:
    for call in r2.tool_calls:
        res = tool_map[call['name']].invoke(call['args'])
        msgs2.append(ToolMessage(content=str(res), tool_call_id=call['id']))
unstructured = llm_with_tools2.invoke(msgs2).content
print(f'Unstructured (string): {unstructured}')
print(f'Type: {type(unstructured)}  ← hard to process in code\n')

# Structured — from Concept 3
_, used3, results3 = execute_tools(q)
structured = synthesis_chain.invoke({
    'query': q, 'tool_results': '\n'.join(results3), 'tools_used': str(used3)
})
print(f'Structured (Pydantic):')
print(f'  .answer     = {structured.answer}')
print(f'  .tools_used = {structured.tools_used}')
print(f'  .confidence = {structured.confidence}')
print(f'  Type: {type(structured).__name__}  ← directly usable in code')

### Full Function

In [ ]:
def structured_agent(query: str) -> str:
    _, used, results = execute_tools(query)
    response = synthesis_chain.invoke({
        'query':        query,
        'tool_results': '\n'.join(results) if results else 'No tools used — answered from knowledge.',
        'tools_used':   str(used),
    })
    # Return just the answer string for run_queries compatibility
    return f'{response.answer}  [tools={response.tools_used}, confidence={response.confidence}]'

### Test — Standard Queries

In [ ]:
run_queries(structured_agent, label='Concept 3 — Structured Output Agent')